# Chapter 7: Finetuning To Follow Instructions

In [1]:
from importlib.metadata import version, PackageNotFoundError

# Used to check installed package versions and handle missing packages


pkgs = ["numpy", "matplotlib", "torch", "tensorflow", "tqdm", "tiktoken"]

for p in pkgs:
    try:
        print(f"{p} version: {version(p)}")
    except PackageNotFoundError:
        print(f"{p} is NOT installed")

numpy version: 2.0.2
matplotlib version: 3.10.6
torch version: 2.9.1+cpu
tensorflow version: 2.20.0
tqdm version: 4.67.3
tiktoken version: 0.12.0


### 7.1 Introduction to instruction finetuning

In chapter 5, we saw that pretraining an LLM involves a training procedure where it learns to generate one word at a time
Hence, a pretrained LLM is good at text completion, but it is not good at following instructions
In this chapter, we teach the LLM to follow instructions better

----------------------------------------------------------------------------------------------------------

Instruction:

Summarize the following text in one sentence:

“Artificial Intelligence enables machines to learn from data, identify patterns, and make decisions with minimal human intervention.”

Desired response:

Artificial Intelligence allows machines to learn from data and make decisions automatically.

-------------------------------------------------------------------------------------------------------------------------------------

### 7.2 Preparing a dataset for supervised instruction finetuning

In [2]:
import json # Used to read JSON files and convert them into Python objects (list/dict).
import os # Used to check whether a file already exists on our system.
import requests  # Used to send an HTTP request to download the file from the internet.


def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        response = requests.get(url, timeout=30)
        response.raise_for_status()  # 404 error, server error, Network issue
        #Know more about 404 error: https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Status/404
        
        text_data = response.text
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return data

# params
file_path = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)

data = download_and_load_file(file_path, url)

print("Number of entries:", len(data))

Number of entries: 1100


In [10]:
data[1]

{'instruction': 'Edit the following sentence for grammar.',
 'input': 'He go to the park every day.',
 'output': 'He goes to the park every day.'}

In [9]:
print(data[1].keys())
print(data[1]['instruction'])
print(data[1]['input'])
print(data[1]['output'])


dict_keys(['instruction', 'input', 'output'])
Edit the following sentence for grammar.
He go to the park every day.
He goes to the park every day.


In [11]:
data[2]

{'instruction': 'Convert 45 kilometers to meters.',
 'input': '',
 'output': '45 kilometers is 45000 meters.'}

In [12]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

In [13]:
print(format_input(data[1]))

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Edit the following sentence for grammar.

### Input:
He go to the park every day.


In [22]:
model_input = format_input(data[2])
desired_response = f"\n\n### Response:\n{data[2]['output']}"

print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Convert 45 kilometers to meters.

### Response:
45 kilometers is 45000 meters.


In [25]:
train_portion = int(len(data) * 0.85)  # 85% for training
test_portion = int(len(data) * 0.1)    # 10% for testing
val_portion = len(data) - train_portion - test_portion  # Remaining 5% for validation

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

In [26]:
print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 935
Validation set length: 55
Test set length: 110


### 7.3 Organizing data into training batches

In [27]:
import torch
from torch.utils.data import Dataset


class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        # Pre-tokenize texts
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)